In [1]:
# Generating normal baseline metrics
import numpy as np

rng = np.random.default_rng(seed=42)

# simulate 1 hour of CPU reading - one per minute
# normal server runs at 60% with small fluctuations
cpu = rng.normal(loc=60, scale=5, size = 60)
#cpu = rng.normal(60,5,60) # mean, std, size
cpu = np.clip(cpu, 0, 100) # cpu cannot exceed 0-100 range

# loc is the mean, scale is the standard deviation. clip keeps values inside a valid range.
print(cpu.round(1))
print(f"mean:{cpu.mean():.1f} std: {cpu.std():.1f}")

[61.5 54.8 63.8 64.7 50.2 53.5 60.6 58.4 59.9 55.7 64.4 63.9 60.3 65.6
 62.3 55.7 61.8 55.2 64.4 59.8 59.1 56.6 66.1 59.2 57.9 58.2 62.7 61.8
 62.1 62.2 70.7 58.  57.4 55.9 63.1 65.6 59.4 55.8 55.9 63.3 63.7 62.7
 56.7 61.2 60.6 61.1 64.4 61.1 63.4 60.3 61.4 63.2 52.7 58.4 57.6 56.8
 58.6 67.5 55.7 64.8]
mean:60.3 std: 3.9


In [12]:
# Generating multiple metrics
rng = np.random.default_rng(seed = 42)
n = 60 # 60 minutes

# each metric has its own releastic baseline and spread
cpu = np.clip(rng.normal(60,5,n), 0, 100)
memory = np.clip(rng.normal(70,8,n), 0, 100)
disk_io = np.clip(rng.normal(40,15,n), 0, 100)
latency = np.clip(rng.normal(15,3,n), 0, 500)

# stack into one dataset - shape (60,4)
metrics = np.column_stack ([cpu, memory, disk_io, latency])
print(metrics)
print(metrics.shape)
print(metrics.round(1)[:5]) # first 5 rows
#each row is one minute. each column is one metric

[[61.5235854  56.53704183 24.64753761 18.91800523]
 [54.80007947 67.32091976 42.68913452 15.6581475 ]
 [63.75225598 71.30202452 43.29995026 13.76721831]
 [64.70282358 74.68977865 60.38781363 18.31886613]
 [50.24482406 75.68981264 52.52666869 16.28626932]
 [53.48910247 76.34677788 45.35306589 19.60726798]
 [60.63920202 67.21019942 61.94954337 15.54970331]
 [58.41878704 66.30118566 22.16855419 11.3265929 ]
 [59.91599421 76.86380705 30.40372701 10.8955224 ]
 [55.73478036 68.4695654  26.10136088 19.9527838 ]
 [64.39698987 59.79450941 34.15285295 20.17099716]
 [63.88895968 60.93370229 19.34970779 14.46144236]
 [60.33015349 62.64438171 49.5272642  13.85043804]
 [65.63620603 73.97728595 36.66665954 19.38433288]
 [62.33754671 71.13940589 17.93790558 11.67886295]
 [55.70353769 75.52388283 24.76631378 12.31581894]
 [61.84375392 66.58197883 44.70270771 16.92998038]
 [55.205587   71.26831753 52.57189852 13.81618463]
 [64.39225151 75.00472315 69.95096338 14.9846344 ]
 [59.75037045 67.52522768 83.70

In [4]:
# Injecting anomalies - CPU spikes
import numpy as np
rng = np.random.default_rng(seed = 42)
cpu = np.clip(rng.normal(60, 5, 60), 0, 100)

#Inject 3 random spikes between 88 and 98 %
spike_idx = rng.choice(60, size=3, replace=False)
#rng.choice picks random indices without replacement - no two spikes at the same minute.
cpu[spike_idx] = rng.uniform(88, 98, size=3) # low, high and size(no of random values)
print(f"spike indices: {spike_idx}")
print(f"spike values: {cpu[spike_idx].round(1)}")
print(cpu.round(0))
print(f"mean: {cpu.mean():.1f} max: {cpu.max():.1f}")

spike indices: [49 33  4]
spike values: [89.1 89.1 88.6]
[62. 55. 64. 65. 89. 53. 61. 58. 60. 56. 64. 64. 60. 66. 62. 56. 62. 55.
 64. 60. 59. 57. 66. 59. 58. 58. 63. 62. 62. 62. 71. 58. 57. 89. 63. 66.
 59. 56. 56. 63. 64. 63. 57. 61. 61. 61. 64. 61. 63. 89. 61. 63. 53. 58.
 58. 57. 59. 67. 56. 65.]
mean: 62.0 max: 89.1


In [8]:
# Injecting correlated anomalies - realistic incidents
# In real incidents, CPU spike causes latency spike. They happen together
import numpy as np
rng = np.random.default_rng(seed=42)
n = 60
cpu = np.clip(rng.normal(60,5,n), 0, 100)
latency = np.clip(rng.normal(15, 3, n), 0, 500)

# Incident at minutes 20-23 - cpu and latency spike together
incident_idx = np.arange(20, 24) # sequence start, stop
cpu[incident_idx] = rng.uniform(88, 98, size=4)
latency[incident_idx] = rng.uniform(80, 120, size=4)

print("cpu at incident:   ", cpu[incident_idx].round(1))
print("latency at incident : ", latency[incident_idx].round(1))
print("CPU normal mean  : ", np.delete(cpu, incident_idx).mean().round(1))
print("Latency normal mean :", np.delete(latency, incident_idx). mean().round(1))
#np.delete removes the incident minutes so you can calculate what normal looks like without them.

cpu at incident:    [88.8 92.2 88.4 92.9]
latency at incident :  [ 93.2  85.8  84.1 103.5]
CPU normal mean  :  60.3
Latency normal mean : 14.5


In [11]:
# simulating multiple servers
import numpy as np
rng = np.random.default_rng(seed =42)
n = 60 # 60 minutes
#3 servers with different baseline behavious
server_a = np.clip(rng.normal(60, 5, n), 0, 100)  #healthy
server_b = np.clip(rng.normal(75, 8, n), 0, 100)  #running hot
server_c = np.clip(rng.normal(40, 3, n), 0, 100) #low utilisation

# stack as row - shape (3,60)
servers = np.vstack([server_a, server_b, server_c])
print(servers)
print(servers.shape)
print(servers.mean(axis=1).round(1)) 
print(servers.std(axis=1).round(1))

#output observation: server B is running hot at 73.5% average 
# server c is barely use. server A is healthy

[[61.5235854  54.80007947 63.75225598 64.70282358 50.24482406 53.48910247
  60.63920202 58.41878704 59.91599421 55.73478036 64.39698987 63.88895968
  60.33015349 65.63620603 62.33754671 55.70353769 61.84375392 55.205587
  64.39225151 59.75037045 59.07568818 56.59535228 66.11270669 59.22735259
  57.85836089 58.23933225 62.66154593 61.82722032 62.06366306 62.15410502
  70.708238   57.96792492 57.43878635 55.93113636 63.07989711 65.64486146
  59.43026271 55.79921762 55.87759392 63.25296394 63.71627086 62.71577134
  56.67245146 61.16080662 60.58342905 61.09344298 64.35714389 61.11797774
  63.39456782 60.33789535 61.44559699 63.15644113 52.7142209  58.40164392
  57.64813673 56.80561076 58.62428874 67.47470656 55.67084442 64.84139177]
 [61.53704183 72.32091976 76.30202452 79.68977865 80.68981264 81.34677788
  72.21019942 71.30118566 81.86380705 73.4695654  64.79450941 65.93370229
  67.64438171 78.97728595 76.13940589 80.52388283 71.58197883 76.26831753
  80.00472315 72.52522768 78.6542019  6

In [13]:
# Dataset - working all together
import numpy as np
rng = np.random.default_rng(seed=99)
n=120 # 2 hours of data, one reading per minute
# generate cpu readings - mean 65, std 6, clipped 0-100
cpu = np.clip(rng.normal(65, 6, n), 0, 100)
print(f"cpu mean: {cpu.mean():.1f} std: {cpu.std():.1f} ")

# generate memory readings - mean 72, std 10, clipped 0-100
memory = np.clip(rng.normal(72, 10, n), 0, 100)
print(f"memory mean: {memory.mean():.1f}  std: {memory.std():.1f} ")

#generate latency readings - mean 20, std 4, clipped 0-500
latency = np.clip(rng.normal(20, 4, n), 0, 500)
print(f"latency mean : {latency.mean():.1f} std: {latency.std():.1f}")

#Inject 4 cpu spikes at random indices - values between 90 and 99
spike_idx = rng.choice(n, size=4, replace=False)
cpu[spike_idx] = rng.uniform(90, 99, size=4)
print(f"spikes at:  {spike_idx} ")
print(f"spike values: {cpu[spike_idx].round(1)}")

# Inject a coorelated incident at minutes 60-63
# cpu goes to 88-95, latency goes to 100-150

incident = np.arange(60, 64)
cpu[incident] = rng.uniform(88, 95, size=4)
latency[incident] = rng.uniform(100, 150, size=4)
print(f"cpu at incident:  {cpu[incident].round(1)}")
print(f"latency at incident: {latency[incident].round(1)} ")

#stack cpu, memory, latency into one dataset shape (120,3)
dataset = np.column_stack([cpu, memory, latency])
print(dataset)
print(dataset.shape)

# mean and std for each metric across all 120 minutes
print(dataset.mean(axis=0). round(1))
print(dataset.std(axis=0).round(1))


cpu mean: 65.1 std: 5.5 
memory mean: 72.5  std: 10.2 
latency mean : 19.9 std: 4.5
spikes at:  [14 44 51 21] 
spike values: [94.9 99.  96.9 92.5]
cpu at incident:  [94.9 90.2 93.3 90.6]
latency at incident: [136.3 123.6 123.9 118.6] 
[[ 65.49496583  68.20888942  21.86370578]
 [ 62.21348951  85.36551235  19.01092408]
 [ 65.30309038  78.89122127  24.42313084]
 [ 69.11738492  75.32664784  20.4028674 ]
 [ 54.45925697  54.85914065  16.66029905]
 [ 75.10658961  65.8239017   15.00501461]
 [ 62.25294296  74.67985771  16.299998  ]
 [ 61.42147943  65.56154848  19.86079275]
 [ 58.71819463  74.82144612  19.14675391]
 [ 70.59075214  74.99291493   9.12187273]
 [ 69.0498829   63.68747259  22.17021258]
 [ 72.46664721  60.86282045  26.37130405]
 [ 70.35852453  81.67495613  17.2243195 ]
 [ 66.57802966  72.48326692  21.86030838]
 [ 94.88899375  72.54981913  16.91275702]
 [ 70.61146216  81.61437475  25.2085739 ]
 [ 59.73432204  89.33206474  16.80910584]
 [ 64.72462347  43.90648629  19.28416223]
 [ 67.291